In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "volter2014great")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Voelter_2014_pilot1_JCompPsychol_GROD.csv")
complete_path_2 = os.path.join(original_data_pathway, "Voelter_2014_pilot2_JCompPsychol_GROD.csv")
complete_path_3 = os.path.join(original_data_pathway, "Voelter_2014_exp1_JCompPsychol_GROD.csv")
complete_path_4 = os.path.join(original_data_pathway, "Voelter_2014_exp2_JCompPsychol_GROD.csv")
complete_path_5 = os.path.join(original_data_pathway, "Voelter_2014_exp3_JCompPsychol_GROD.csv")
complete_path_6 = os.path.join(original_data_pathway, "Voelter_2014_exp4_JCompPsychol_GROD.csv")
complete_path_7 = os.path.join(original_data_pathway, "Voelter_2014_exp5_JCompPsychol_GROD.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)
experiment_import = [[df1, 'pilot_1','0'],
                    [df2, 'pilot_2', '0']]
for x,y,k in experiment_import: 
    x['experiment_name']=y
    x['experiment']=k
    
df3 = pd.read_csv(complete_path_3)
df3 = df3.assign(experiment='1')
df4 = pd.read_csv(complete_path_4)
df4 = df4.assign(experiment='2')
df5 = pd.read_csv(complete_path_5)
df5 = df5.assign(experiment='3')
df6 = pd.read_csv(complete_path_6)
df6 = df6.assign(experiment='4')
df7 = pd.read_csv(complete_path_7)
df7 = df7.assign(experiment='5')


In [3]:
df3.columns
df3['Trial_type'].replace('Base', 'control', inplace=True, regex=True)
df4['Trial_type'].replace('Base', 'control', inplace=True, regex=True)
df5['Trial_type'].replace('Base', 'two_trail_control', inplace=True, regex=True)
df6['Trial_type'].replace('old', 'preexisting', inplace=True, regex=True)
df6['Trial_type'].unique()


array(['preexisting', 'recent'], dtype=object)

In [4]:
df6.columns

Index(['Species', 'Name', 'Age', 'Group', 'Session', 'Trial_number',
       'Trial_type', 'Location', 'Reward', 'experiment'],
      dtype='object')

In [5]:
data_frames=[df1, df2, df3, df4, df5, df6, df7]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"subject": "ape",
        "name":"ape",
        "species":"species_original", 
        "group(trace/notrace)":"group_original",
        "group":"group_original",
        "trial#":"trial_number",
        "trial type":"trial_type",
        "reward (incorrect=0/correct=1)":"reward",}, inplace=True)
    x['ape'] = x['ape'].str.rstrip()
    x['study_id']="volter2014great"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [6]:
code_list=["reward"]
for index, x in enumerate(code_list):    
    fulldf[x] = fulldf[x].astype(str)
    temp=[]
    for entry in fulldf[x]:
        if entry == '0' or entry == '0.0':
            entry = "incorrect"
        elif entry =='1' or entry == '1.0':
            entry = "correct"
        temp.append(entry)
    fulldf = fulldf.assign(temp_col=temp)
    fulldf=fulldf.rename(columns={'temp_col': x+'_codes'})

In [7]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')
fulldf.rename(columns={"ape": "participant", "age":"age_in_years"}, inplace=True)
# fulldf.columns

In [8]:
fulldf = fulldf[['study_id', 'experiment', 'experiment_name','participant', 'age_in_years','sex','species',
        'session','trial_number', 'trial_type', 
        'location', 'reward', 
         'order', 'selected_location',
       'reward_codes' ]]

fulldf.dropna(subset=['participant'], inplace=True)


In [9]:
for index in range(0,6):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'volter2014great_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'volter2014great_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)